Import the libraries

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch
from torchvision import transforms
from PIL import Image

Create the CNN

In [ ]:
class ImageClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
    self.pool = nn.MaxPool2d(2)
    self.flatten = nn.Flatten()
    self.fc1 = nn.Linear(8 * 56 * 56, 32)
    self.fc2 = nn.Linear(32, 2)

  def forward(self, x):
    out = self.pool(F.relu(self.conv1(x)))
    out = self.pool(F.relu(self.conv2(out)))
    out = self.flatten(out)
    out = self.fc2(F.relu(self.fc1(out)))
    return out

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ImageClassifier().to(device)

Load the training data

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Load the images for the training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder('./dataset/train', transform=transform)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)

val_data = datasets.ImageFolder('./dataset/val', transform=transform)
val_loader = DataLoader(val_data, batch_size=16, shuffle=True)

Loop to:

1. Feed the images in the model
2. Calculate how far off the prediction is
3. Adjust the model's weights to be more acccurate

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader

# Loss Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

history = {"val_loss": [], "val_acceptance": [], "train_loss": [], "train_acceptance": []}

# Train Loop
for epoch in range(16):
  # Training phase
  model.train()
  running_loss = 0.0
  train_correct = 0
  train_total = 0
  
  for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)
      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      _, predicted = torch.max(outputs.data, 1)
      train_total = labels.size(0)
      train_correct = (predicted == labels).sum().item()

  epoch_train_loss = running_loss / len(train_loader)
  epoch_train_acc = 100 * train_correct / train_total


  # Validation phase
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for images, labels in val_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()
  # print(f"Epoch {epoch+1}/16 | Loss: {running_loss/len(9):.4f} | Val Acc: {100 * correct / total:.2f}%")

  epoch_val_loss = running_loss / len(val_loader)
  epoch_val_acc = 100 * correct / total

  history["val_loss"].append(epoch_val_loss)
  history["val_acceptance"].append(epoch_val_acc)
  history["train_loss"].append(epoch_train_loss)
  history["train_acceptance"].append(epoch_train_acc)

# Save the weights
torch.save(model.state_dict(), 'leaderboard_model.pth')

Validation vs Training plots

In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(16), history["train_loss"], label="Training Loss", linewidth=2)
plt.plot(range(16), history['val_loss'], label="Validation Loss", linewidth=2)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(range(16), history["train_acceptance"], label="Training Acceptance", linewidth=2)
plt.plot(range(16), history['val_acceptance'], label="Validation Acceptance", linewidth=2)
plt.xlabel("Epochs")
plt.ylabel("Acceptance")
plt.legend()
plt.show()